# Part 1: Dogs vs Cats Under Tile Orders
This notebook trains three image-classification architecture families on Dogs vs Cats.
It evaluates how fixed tile-wise tile permutations affect validation accuracy.
Results are aggregated by tile count and plotted as accuracy vs number of tiles.


## Setup

### Global / External Imports
Import third-party libraries used only for notebook orchestration and display.


most basic imports

In [ ]:
from pathlib import Path
import importlib
import os
import sys


### Local Imports
Import project modules. Part 1 orchestration lives in this notebook; reusable training, preprocessing, and evaluation helpers stay in `src`.


prep for local imports

In [ ]:
_BOOTSTRAP_ROOT = None
_current = Path.cwd().resolve()
for _candidate in [_current, *_current.parents]:
    if (_candidate / 'src').is_dir() and (_candidate / 'requirements.txt').exists():
        _BOOTSTRAP_ROOT = _candidate
        break

if _BOOTSTRAP_ROOT is None:
    _drive_root = Path('/content/drive/MyDrive/MLDS_Final_Project')
    if _drive_root.exists():
        _BOOTSTRAP_ROOT = _drive_root

if _BOOTSTRAP_ROOT is not None and str(_BOOTSTRAP_ROOT) not in sys.path:
    sys.path.insert(0, str(_BOOTSTRAP_ROOT))

from src.utils.colab import bootstrap_notebook_runtime  # noqa: E402

ROOT = bootstrap_notebook_runtime(_BOOTSTRAP_ROOT)
ROOT


make local imports

In [ ]:
import pandas as pd
from IPython.core.display import Image
from IPython.display import display

import src.evaluation.experiment_results as experiment_results

experiment_results = importlib.reload(experiment_results)
get_device = experiment_results.get_device
from src.utils.colab import print_colab_runtime_diagnostics, warn_if_colab_runtime_without_cuda  # noqa: E402
from src.utils.reproducibility import seed_everything  # noqa: E402


### Setup configs

In [ ]:
import src.utils.config as config_module
import src.models.factory as model_factory
import src.experiments.part1 as part1_experiments

config_module = importlib.reload(config_module)
model_factory = importlib.reload(model_factory)
part1_experiments = importlib.reload(part1_experiments)

CVExperimentConfig = config_module.CVExperimentConfig
experiment_output_paths = experiment_results.experiment_output_paths
load_experiment_samples = experiment_results.load_experiment_samples
plot_accuracy_vs_tiles = experiment_results.plot_accuracy_vs_tiles
save_aggregated_accuracy = experiment_results.save_aggregated_accuracy
save_rows = experiment_results.save_rows
train_model_on_tile_permutation_records = part1_experiments.train_model_on_tile_permutation_records
from src.preprocessing.samples import class_counts  # noqa: E402
from src.preprocessing.tile_permutations import build_tile_permutation_records, tile_permutation_to_jsonable  # noqa: E402
from src.utils.io import save_csv  # noqa: E402
from src.utils.plotting import plot_tile_permutation_samples  # noqa: E402


In [ ]:
configs = CVExperimentConfig()
device = get_device(config=configs)
warn_if_colab_runtime_without_cuda(device)
print_colab_runtime_diagnostics(device)
display(configs)


Local runs use a balanced 256-image subset and 5 training epochs. With `val_fraction=0.2`, the validation split is large enough for accuracy to move in smaller steps than the old 6-image smoke-test split. In aggregated results, `final_epoch` means the metric from the last epoch, and `best_epoch` means the best validation score observed during training.

### make installations before final external imports

In [ ]:
# Dependencies are installed during the setup/bootstrap cell above when running in Colab.
print('Dependency setup is complete.')


### final imports (after doing pip install if working on colab)

In [ ]:
import json
import random
from IPython.core.display import Image
from IPython.display import display
import pandas as pd
import numpy as np
import torch


## Experiments

### Experiment helpers
Shared helpers are kept in `src/evaluation/experiment_results.py`; notebook-specific helpers stay here.

In [ ]:
def get_part1_output_paths(*, config: CVExperimentConfig) -> dict[str, str]:
    """Build stable Part 1 output paths for notebook display."""
    paths = experiment_output_paths(
        results_dir=config.results_dir,
        figures_dir=config.figures_dir,
        part_name='part1',
    )
    paths['accuracy_plot'] = paths['figure']
    return paths

### Setup Exp

In [ ]:
# 4. Reproducibility (important for multiple notebooks
random.seed(configs.seed)
np.random.seed(configs.seed)
torch.manual_seed(configs.seed)
torch.set_num_threads(configs.max_threads)  # avoid contention across notebooks

In [ ]:
output_paths = get_part1_output_paths(config=configs)
output_paths

### Data Loading
Discover the configured Dogs vs Cats split before training.


In [ ]:
train_samples, validation_samples, test_samples = load_experiment_samples(config=configs, seed=configs.seed)
print(f'Train samples: {len(train_samples)}')
print(f'Validation samples: {len(validation_samples)}')
print(f'Test samples: {len(test_samples)}')
print('Train class counts:', class_counts(samples=train_samples))
print('Validation class counts:', class_counts(samples=validation_samples))
print('Test class counts:', class_counts(samples=test_samples))

### Experiments - Run Baselines
Run the configured baseline grid/model/tile permutation sweep. Re-run this cell to regenerate Part 1 outputs.


In [ ]:
# Build and save tile permutation records
tile_permutation_records = build_tile_permutation_records(
    tiles_per_side_values=configs.tiles_per_side_values,
    num_tile_permutations=configs.num_tile_permutations,
    seed=configs.seed,
    include_baseline=True,
)
tile_permutation_rows = [record.__dict__ | {'tile_permutation': json.dumps(tile_permutation_to_jsonable(record.tile_permutation))} for record in tile_permutation_records]
save_csv(data=tile_permutation_rows, path=output_paths['tile_permutations'])
print(f"Saved {len(tile_permutation_records)} tile permutation records")

if configs.plot_samples:
    import matplotlib.pyplot as plt

    plot_tile_permutation_samples(
        samples=train_samples,
        tile_permutation_records=tile_permutation_records,
        image_size=configs.image_size,
    )
    plt.show()


In [ ]:
# Setup reproducibility and load data
seed = configs.seed
seed_everything(seed=seed, deterministic=configs.deterministic)
# Data already loaded above
print(f"Loaded {len(train_samples)} train, {len(validation_samples)} val samples")

In [ ]:
# Prepare shared result accumulation across model runs
all_rows = []
run_id = configs.config_name

### Train Lightweight Model Trio
Train the configured pretrained trio: ResNet-18, DeiT-Tiny, and MLP-Mixer Small.

In [ ]:
for model_name in configs.model_names:
    rows = train_model_on_tile_permutation_records(
        config=configs,
        model_name=model_name,
        run_id=run_id,
        train_samples=train_samples,
        validation_samples=validation_samples,
        tile_permutation_records=tile_permutation_records,
        seed=seed,
        device=device,
        raw_results_output_path=output_paths["raw_results"],
    )
    all_rows.extend(rows)
    print("Completed training", model_name, "with", len(rows), "runs")


In [ ]:
# Aggregate results and plot
raw_results = pd.read_csv(filepath_or_buffer=output_paths['raw_results'])
aggregated_results = save_aggregated_accuracy(
    raw_results=raw_results,
    group_columns=['model_name', 'tiles_per_side', 'num_tiles'],
    output_path=output_paths['aggregated_results'],
)
plot_accuracy_vs_tiles(aggregated=aggregated_results, output_path=output_paths['accuracy_plot'])
aggregated_results

### Experiments - Results Table
Reload the saved aggregated CSV so the report is reproducible from disk. Each row averages all tile permutation scores for the same model and tile count.


In [ ]:
saved_results = {
    'raw': pd.read_csv(filepath_or_buffer=output_paths['raw_results']),
    'aggregated': pd.read_csv(filepath_or_buffer=output_paths['aggregated_results']),
}
display(saved_results['aggregated'])

### Experiments - Accuracy Plot
Display the saved accuracy-vs-number-of-tiles plot.


In [ ]:
figure_path = output_paths['accuracy_plot']
if Path(figure_path).exists():
    display(Image(filename=figure_path))
else:
    print(f'Plot not found yet: {figure_path}')


In [ ]:
# Export this saved notebook to PDF. Save the notebook before running this cell,
# because nbconvert reads the on-disk .ipynb file rather than unsaved editor state.
import subprocess

notebook_path = Path(REPO_PATH) / 'src' / 'notebooks' / 'part1_solution.ipynb'
export_dir = Path(REPO_PATH) / 'outputs' / 'notebooks'
export_dir.mkdir(parents=True, exist_ok=True)

try:
    subprocess.run(
        [
            sys.executable,
            '-m',
            'jupyter',
            'nbconvert',
            '--to',
            'pdf',
            str(notebook_path),
            '--output-dir',
            str(export_dir),
        ],
        check=True,
    )
    print(f'Generated PDF: {export_dir / notebook_path.with_suffix(".pdf").name}')
except subprocess.CalledProcessError as exc:
    print('PDF export failed. Confirm the notebook is saved and that nbconvert plus LaTeX are installed.')
    print(f'Command exited with status {exc.returncode}.')
